# 4 · Results: reproduction vs. paper

This notebook gathers the cached runs (created by notebook 3 or by
`python scripts/run_experiments.py --fv FV1 FV2`) and compares them with the paper's
Tables 10–13 and Figures 3–5.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make `lora_loc` importable from notebooks/
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lora_loc import config, data, preprocessing as pp, features, geo, plotting
plotting.style()
pd.set_option("display.precision", 2)
from lora_loc import models, experiment, paper_results as paper
res = experiment.collect()
res = res.set_index(["fv", "model"]).sort_index()
available_fvs = sorted(res.index.get_level_values(0).unique(), key=lambda s: int(s[2:]))
print("feature vectors with results:", available_fvs)

## 4.1 Mean error per feature vector (Tables 10–11)

In [ ]:
rows = []
for fv in available_fvs:
    for name in models.MODEL_NAMES:
        if (fv, name) in res.index:
            ours = res.loc[(fv, name), "mean_m"]; ref = paper.MEAN_ERROR_M.loc[name, fv]
            rows.append({"FV": fv, "model": name, "ours (m)": ours, "paper (m)": ref,
                         "Δ (m)": ours - ref, "Δ (%)": 100 * (ours - ref) / ref})
cmp = pd.DataFrame(rows).set_index(["FV", "model"])
display(cmp.round(2))
print("mean |Δ| per FV (m):"); print(cmp["Δ (m)"].abs().groupby(level=0).mean().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, len(available_fvs), figsize=(7.5 * len(available_fvs), 4), squeeze=False)
for ax, fv in zip(axes[0], available_fvs):
    ours = res.loc[fv, "mean_m"]
    plotting.compare_bars(ours, paper.MEAN_ERROR_M[fv], ax=ax, title=f"{fv}: mean error")
    ax.set_ylim(0, max(ours.max(), 350) * 1.15)
plt.tight_layout(); plt.show()

## 4.2 Error distributions (Fig. 3, Table 12)

In [ ]:
best_fv = available_fvs[-1]
errs = {name: experiment.run(name, best_fv, verbose=False)["errors_m"]
        for name in models.MODEL_NAMES if (best_fv, name) in res.index}
plotting.cdf(errs, xmax=1500, thresholds=config.CDF_THRESHOLDS_M)
plt.title(f"Error CDF on {best_fv}"); plt.show()

cols = [f"within_{t}m" for t in config.CDF_THRESHOLDS_M]
tab = res.loc[best_fv, cols].copy(); tab.columns = [f"≤{t} m" for t in config.CDF_THRESHOLDS_M]
print(f"Reproduced, {best_fv} (fraction of test samples):"); display((tab * 100).round(0).astype(int))
print("Paper Table 12 (best FV per model, mostly FV4/FV15/FV16 — not directly comparable):")
display((paper.CDF_TABLE12 * 100).round(0).astype(int))

Summary statistics. The paper's Eq. 20 SD is the population SD of the per-sample errors.

In [ ]:
stats = res[["mean_m", "median_m", "sd_m", "p90_m", "max_m", "r2_lat", "r2_lon"]]
display(stats.round(3))

## 4.3 Where are the errors? (Fig. 4)

In [ ]:
best = res.loc[best_fv, "mean_m"].idxmin()
r = experiment.run(best, best_fv, verbose=False)
ax, share = plotting.error_heatmap(r["y_true_deg"], r["errors_m"], cell_m=150)
ax.set_title(f"{best} on {best_fv}: mean error per 150 m cell"); plt.show()
print({k: f"{v:.1%}" for k, v in share.items()})
print("paper (Hybrid, best FV): 21.7% of cells < 250 m, 33% < 300 m, 42.6% < 350 m")

Large errors cluster at the edges of the covered area and along sparsely driven roads, where
few training fingerprints exist and messages are often heard by a single gateway.

## 4.4 Inference time (Fig. 5)
Per-sample prediction time, measured by predicting one sample at a time on 200 random test
samples. Absolute numbers depend on the hardware. Our CPU is not the authors' machine, so compare
the **ranking**, not the values.

In [ ]:
t = res.loc[best_fv, ["infer_single_mean_s", "infer_batched_s", "train_s"]].copy()
t["paper per-sample (s)"] = pd.Series(paper.INFERENCE_S)
display(t.rename(columns={"infer_single_mean_s": "ours per-sample (s)",
                          "infer_batched_s": "ours batched, per sample (s)",
                          "train_s": "ours training (s)"}).round(5))

## 4.5 FV3–FV16 (needs the JSON release)

In [ ]:
if config.JSON_FILE.exists():
    for fv in [f"FV{i}" for i in range(3, 17)]:
        d = experiment.prepare_fv(fv)
        for name in ["k-NN", "XGBoost", "LightGBM"]:   # add more models if you have the time
            experiment.run(name, fv, prepared=d)
    print("Re-run this notebook from the top to include FV3-FV16 in the tables above.")
else:
    print(f"{config.JSON_FILE.name} not found -> FV3-FV16 skipped.\n"
          "Download it with `python scripts/download_data.py --json`, then re-run this cell.")

## 4.6 Discussion: what reproduces, what does not, and why

See the **Reproduction report** section of the repository `README.md` for the final numbers
from our run and a full list of deviations. The key points for students:

1. **The protocol reproduces exactly.** Dataset checksum, 44 active gateways, −200 → −128 dBm,
   and split sizes (91,300 / 19,564 / 19,565) all match the paper.
2. **Mean errors on FV1/FV2 are close for the non-neural models,** with small differences. Remaining
   gaps come from unreported details (SVR γ, ANN learning rate, hybrid kernel size / T), random
   initialisation of the neural networks, and library versions.
3. **Differences between the tree models are within a few metres,** which is about the size of the
   run-to-run variation of the neural models. Rankings between close models should not be
   over-interpreted without repeated runs (different seeds) and confidence intervals.
4. **RSSI-only fingerprinting in this dataset has a floor of roughly 300 m on average**, set by
   sparse coverage (many messages heard by one gateway). The paper's gains to ≈245 m come from
   FVs with the **frame counter**, which, under a random per-message split, may encode
   track continuity rather than radio information that transfers to new devices (see notebook 2).
5. **Median ≪ mean:** the error distribution has a heavy tail, so report median, 90th percentile
   and the CDF, not just the mean.